# People counting in photos and videos (MiVOLO)

**How to use:**
1. Menu `Runtime > Change runtime type` and pick **GPU (T4)**
2. Run cells 1, 2 and 3 (once per session)
3. Use cell 4 to count from **photos** or cell 5 to count from **video** (20-30s pan over the room)

The history is saved to `My Drive/people_counts/counts.csv`, model weights are cached in your Drive and **uploaded photos/videos are deleted at the end** (only aggregated numbers are kept).

**Video tip:** record at 720p, slow and continuous motion, sweeping the room a single time. Long videos are slow (~1 min of processing per 10s of video).

In [ ]:
#@title 1) Install dependencies (run once per session, ~3 min)
!pip install -q git+https://github.com/WildChlamydia/MiVOLO.git
print('Dependencies installed.')

In [ ]:
#@title 2) Mount Google Drive (history + model cache)
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')

BASE_DIR = Path('/content/drive/MyDrive/people_counts')
MODELS_DIR = BASE_DIR / 'models'
MODELS_DIR.mkdir(parents=True, exist_ok=True)
HISTORY_FILE = BASE_DIR / 'counts.csv'

print(f'History: {HISTORY_FILE}')
print(f'Model cache: {MODELS_DIR}')

In [ ]:
#@title 3) Load MiVOLO and shared functions (run once per session)
import csv
import os
import cv2
import gdown
import torch
from argparse import Namespace
from collections import Counter
from dataclasses import dataclass
from datetime import datetime
from google.colab.patches import cv2_imshow
from mivolo.predictor import Predictor

# --- Model weights (cached in Drive after first download) ---
DETECTOR_WEIGHTS = MODELS_DIR / 'yolov8x_person_face.pt'
MIVOLO_CHECKPOINT = MODELS_DIR / 'mivolo_imdb.pth.tar'
DETECTOR_GDRIVE_ID = '1CGNCkZQNj5WkP3rLpENWAOgrBQkUWRdw'
CHECKPOINT_GDRIVE_ID = '11i8pKctxz3wVkDBlWKvhYIh7kpVFXSZ4'

# --- Age classification rules ---
CHILD_MAX_AGE = 12
TEEN_MAX_AGE = 17
ADULT_MAX_AGE = 59

AGE_GROUPS = ('child', 'teen', 'adult', 'senior')
GENDERS = ('M', 'F')


def download_weights_if_missing() -> None:
    if not DETECTOR_WEIGHTS.exists():
        print('Downloading detector (once, kept in Drive)...')
        gdown.download(id=DETECTOR_GDRIVE_ID, output=str(DETECTOR_WEIGHTS), quiet=False)
    if not MIVOLO_CHECKPOINT.exists():
        print('Downloading MiVOLO (once, kept in Drive)...')
        gdown.download(id=CHECKPOINT_GDRIVE_ID, output=str(MIVOLO_CHECKPOINT), quiet=False)


def allow_full_checkpoint_loading() -> None:
    """PyTorch >= 2.6 defaults torch.load to weights_only=True, which rejects
    MiVOLO/ultralytics checkpoints (pickled with model classes). Restoring the
    old behavior is safe here: weights come from the official MiVOLO release.
    """
    original_torch_load = torch.load

    def torch_load_trusted(*args, **kwargs):
        kwargs.setdefault('weights_only', False)
        return original_torch_load(*args, **kwargs)

    torch.load = torch_load_trusted


def load_predictor() -> Predictor:
    config = Namespace(
        detector_weights=str(DETECTOR_WEIGHTS),
        checkpoint=str(MIVOLO_CHECKPOINT),
        device='cuda' if torch.cuda.is_available() else 'cpu',
        with_persons=True,   # body + face: key for accuracy on children
        disable_faces=False,
        draw=True,           # recognize() also returns the annotated image
    )
    return Predictor(config, verbose=False)


def classify_age_group(age: float) -> str:
    if age <= CHILD_MAX_AGE:
        return 'child'
    if age <= TEEN_MAX_AGE:
        return 'teen'
    if age <= ADULT_MAX_AGE:
        return 'adult'
    return 'senior'


def normalize_gender(gender_label: str) -> str:
    return 'M' if gender_label == 'male' else 'F'


@dataclass(frozen=True)
class CountResult:
    source_name: str
    total_people: int
    analyzed_individuals: int
    demographics: Counter


def print_summary(result: CountResult, total_label: str) -> None:
    print('\n============ COUNT SUMMARY ============')
    print(f'{total_label}: {result.total_people}')
    print(f'People with estimated demographics: {result.analyzed_individuals}')
    for age_group in AGE_GROUPS:
        male = result.demographics.get(f'{age_group}_M', 0)
        female = result.demographics.get(f'{age_group}_F', 0)
        if male or female:
            print(f'  {age_group.capitalize():8s} M: {male:3d}  F: {female:3d}')
    print('=======================================')


def append_to_history(result: CountResult, event_label: str) -> None:
    header = (['timestamp', 'event', 'total_people', 'analyzed_individuals']
              + [f'{age_group}_{gender}' for age_group in AGE_GROUPS for gender in GENDERS]
              + ['source'])
    row = ([datetime.now().strftime('%Y-%m-%d %H:%M'), event_label,
            result.total_people, result.analyzed_individuals]
           + [result.demographics.get(f'{age_group}_{gender}', 0)
              for age_group in AGE_GROUPS for gender in GENDERS]
           + [result.source_name])

    needs_header = not HISTORY_FILE.exists()
    with open(HISTORY_FILE, 'a', newline='', encoding='utf-8') as csv_file:
        writer = csv.writer(csv_file)
        if needs_header:
            writer.writerow(header)
        writer.writerow(row)
    print(f'\nHistory updated in Drive: {HISTORY_FILE}')


def delete_files(paths: list[str]) -> None:
    """Privacy: keep only aggregated numbers, never the media files."""
    for path in paths:
        try:
            os.remove(path)
        except OSError:
            pass
    print('Uploaded files were deleted from the session.')


download_weights_if_missing()
allow_full_checkpoint_loading()
predictor = load_predictor()
print('MiVOLO ready.')

In [ ]:
#@title 4) Count from PHOTOS
EVENT_LABEL = 'Sunday morning'  #@param {type:"string"}
SHOW_AGE_PREVIEW = True  #@param {type:"boolean"}

from google.colab import files


def summarize_face_demographics(detected) -> tuple[int, Counter]:
    demographics = Counter()
    analyzed = 0
    for index in detected.get_bboxes_inds('face'):
        age, gender = detected.ages[index], detected.genders[index]
        if age is None or gender is None:
            continue
        analyzed += 1
        demographics[f'{classify_age_group(age)}_{normalize_gender(gender)}'] += 1
    return analyzed, demographics


def analyze_photo(path: str, show_preview: bool = False) -> CountResult | None:
    image = cv2.imread(path)
    if image is None:
        print(f'[error] could not read {path}')
        return None
    detected, annotated_image = predictor.recognize(image)
    if show_preview and annotated_image is not None:
        cv2_imshow(annotated_image)
    analyzed, demographics = summarize_face_demographics(detected)
    return CountResult(
        source_name=Path(path).name,
        total_people=max(detected.n_persons, detected.n_faces),
        analyzed_individuals=analyzed,
        demographics=demographics,
    )


def pick_most_complete_photo(results: list[CountResult]) -> CountResult:
    """Among photos of the same event, the highest count is the most representative."""
    return max(results, key=lambda r: r.total_people)


def run_photo_count(event_label: str, show_preview: bool) -> None:
    print('Select the event photos (you can upload several):')
    photos = list(files.upload().keys())
    results = []
    for path in photos:
        result = analyze_photo(path, show_preview=show_preview)
        if result:
            results.append(result)
            print(f'{result.source_name}: {result.total_people} people, '
                  f'{result.analyzed_individuals} with demographics')
    if not results:
        print('No image processed.')
        return
    event_result = pick_most_complete_photo(results)
    print_summary(event_result, total_label='Total people (best photo)')
    append_to_history(event_result, event_label)
    delete_files(photos)


run_photo_count(EVENT_LABEL, SHOW_AGE_PREVIEW)

In [ ]:
#@title 5) Count from VIDEO (slow pan over the room, 20-30s)
EVENT_LABEL_VIDEO = 'Sunday morning'  #@param {type:"string"}

import statistics
from collections import defaultdict
from google.colab import files

# Tracks shorter than this are likely tracker noise (ID switches, false positives)
MIN_TRACK_SAMPLES = 3


def collect_video_tracks(path: str) -> tuple[dict, dict]:
    """Tracks persons and faces SEPARATELY across frames.

    MiVOLO assigns independent track ids to bodies and faces, so merging them
    (as predictor.recognize_video does) would count each visible person twice.
    """
    person_tracks: dict[int, list] = defaultdict(list)
    face_tracks: dict[int, list] = defaultdict(list)

    capture = cv2.VideoCapture(path)
    if not capture.isOpened():
        raise ValueError(f'Could not open video {path}')
    total_frames = int(capture.get(cv2.CAP_PROP_FRAME_COUNT))
    print(f'Processing {total_frames} frames...')

    frame_index = 0
    while True:
        ok, frame = capture.read()
        if not ok:
            break
        detected = predictor.detector.track(frame)
        predictor.age_gender_model.predict(frame, detected)
        persons, faces = detected.get_results_for_tracking()
        for track_id, sample in persons.items():
            if None not in sample:
                person_tracks[track_id].append(sample)
        for track_id, sample in faces.items():
            if None not in sample:
                face_tracks[track_id].append(sample)
        frame_index += 1
        if frame_index % 100 == 0:
            print(f'  {frame_index}/{total_frames} frames')
    capture.release()
    return person_tracks, face_tracks


def drop_noise_tracks(tracks: dict) -> dict:
    return {track_id: samples for track_id, samples in tracks.items()
            if len(samples) >= MIN_TRACK_SAMPLES}


def summarize_track_demographics(tracks: dict) -> Counter:
    """One person per track: median age, majority-vote gender."""
    demographics = Counter()
    for samples in tracks.values():
        median_age = statistics.median(age for age, _ in samples)
        genders = [gender for _, gender in samples]
        majority_gender = max(set(genders), key=genders.count)
        demographics[f'{classify_age_group(median_age)}_{normalize_gender(majority_gender)}'] += 1
    return demographics


def analyze_video(path: str) -> CountResult:
    person_tracks, face_tracks = collect_video_tracks(path)
    person_tracks = drop_noise_tracks(person_tracks)
    face_tracks = drop_noise_tracks(face_tracks)

    richer_tracks = face_tracks if len(face_tracks) >= len(person_tracks) else person_tracks
    return CountResult(
        source_name=Path(path).name,
        total_people=max(len(person_tracks), len(face_tracks)),
        analyzed_individuals=len(richer_tracks),
        demographics=summarize_track_demographics(richer_tracks),
    )


def run_video_count(event_label: str) -> None:
    print('Select the event video:')
    uploaded = list(files.upload().keys())
    if not uploaded:
        print('No video uploaded.')
        return
    video_path = uploaded[0]
    result = analyze_video(video_path)
    print_summary(result, total_label='Total unique people (tracking)')
    append_to_history(result, event_label)
    delete_files(uploaded)


run_video_count(EVENT_LABEL_VIDEO)

In [ ]:
#@title 6) (Optional) Attendance trend chart
import pandas as pd
import matplotlib.pyplot as plt


def plot_attendance_trend(history: pd.DataFrame) -> None:
    fig, ax = plt.subplots(figsize=(10, 4))
    for event, records in history.groupby('event'):
        ax.plot(records['timestamp'], records['total_people'],
                marker='o', label=event or 'unlabeled')
    ax.set_title('People per event')
    ax.set_ylabel('People')
    ax.legend()
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()


history = pd.read_csv(HISTORY_FILE, parse_dates=['timestamp'])
plot_attendance_trend(history)
history.tail(10)